**Phase 2: Data Cleaning**

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RAW = "../data/raw/global_ads_performance_dataset.csv"
OUT = "../data/processed/marketing_clean.csv"

df = pd.read_csv(RAW, parse_dates=["date"])
print("Raw shape:", df.shape)

Raw shape: (1800, 14)


In [2]:
KEYS = ["date", "platform", "campaign_type", "industry", "country"]
TEXT_COLS = ["platform", "campaign_type", "industry", "country"]
RAW_COLS = ["impressions", "clicks", "ad_spend", "conversions", "revenue"]

# strip stray whitespace in text columns
df[TEXT_COLS] = df[TEXT_COLS].apply(lambda s: s.str.strip())

print("Nulls:", int(df.isna().sum().sum()))
print("Full-row duplicates:", df.duplicated().sum())
print("Duplicate key combinations:", df.duplicated(subset=KEYS).sum())

assert (df[RAW_COLS] >= 0).all().all(), "negative values found"
print("\nZero values per raw column:")
print((df[RAW_COLS] == 0).sum())

Nulls: 0
Full-row duplicates: 0
Duplicate key combinations: 12

Zero values per raw column:
impressions    0
clicks         0
ad_spend       0
conversions    0
revenue        0
dtype: int64


In [3]:
clean = df.drop(columns=["CTR", "CPC", "CPA", "ROAS"]).copy()
clean["month"] = clean["date"].dt.to_period("M").dt.to_timestamp()
clean["quarter"] = clean["date"].dt.to_period("Q").astype(str)
clean = clean.sort_values(["date", "platform"]).reset_index(drop=True)

print(clean.shape)
clean.head()

(1800, 12)


,date,platform,campaign_type,industry,country,impressions,clicks,ad_spend,conversions,revenue,month,quarter
0,2024-01-01,Google Ads,Search,SaaS,India,21702,1000,"3,170.0000",17,"3,670.8800",2024-01-01,2024Q1
1,2024-01-01,TikTok Ads,Display,SaaS,USA,103026,9066,"12,964.3800",278,"46,631.0000",2024-01-01,2024Q1
2,2024-01-02,Meta Ads,Video,EdTech,USA,138639,2204,"3,967.2000",154,"35,803.0700",2024-01-01,2024Q1
3,2024-01-02,Meta Ads,Search,Healthcare,India,163624,4368,"5,591.0400",296,"31,041.3300",2024-01-01,2024Q1
4,2024-01-02,Meta Ads,Display,SaaS,UK,143485,3945,"2,800.9500",187,"30,765.3200",2024-01-01,2024Q1


In [4]:
def safe_div(a, b):
    # denominator 0 -> NaN, never 0 or inf
    return a.div(b.where(b != 0))


def add_metrics(g):
    g = g.copy()
    g["CTR"] = safe_div(g["clicks"], g["impressions"])
    g["CPC"] = safe_div(g["ad_spend"], g["clicks"])
    g["conv_rate"] = safe_div(g["conversions"], g["clicks"])
    g["CAC"] = safe_div(g["ad_spend"], g["conversions"])
    g["ROAS"] = safe_div(g["revenue"], g["ad_spend"])
    return g


total = add_metrics(clean[RAW_COLS].sum().to_frame().T)
total

,impressions,clicks,ad_spend,conversions,revenue,CTR,CPC,conv_rate,CAC,ROAS
0,"185,254,234.0000","7,132,816.0000","11,108,749.0900","326,812.0000","54,183,330.8100",0.0385,1.5574,0.0458,33.9913,4.8775


In [5]:
print("Mean of row-level ROAS (misleading):", round(df["ROAS"].mean(), 4))
print("Median of row-level ROAS:           ", round(df["ROAS"].median(), 4))
print("Aggregate ROAS (sum revenue / sum spend):", round(total["ROAS"].iloc[0], 4))

Mean of row-level ROAS (misleading): 6.4504
Median of row-level ROAS:            4.295
Aggregate ROAS (sum revenue / sum spend): 4.8775


In [6]:
row_roas = safe_div(clean["revenue"], clean["ad_spend"])
top = clean.assign(ROAS=row_roas).nlargest(5, "ROAS")[
    ["date", "platform", "campaign_type", "industry", "country", "ad_spend", "revenue", "ROAS"]
]
print("Top 5 rows by ROAS:")
display(top)
print("Revenue share of these 5 rows:", round(top["revenue"].sum() / clean["revenue"].sum(), 4))

Top 5 rows by ROAS:


,date,platform,campaign_type,industry,country,ad_spend,revenue,ROAS
661,2024-05-07,TikTok Ads,Display,SaaS,UK,"1,660.0500","81,349.3500",49.0042
1786,2024-12-29,TikTok Ads,Display,E-commerce,Germany,"1,094.6100","51,119.7300",46.7013
738,2024-05-23,TikTok Ads,Shopping,EdTech,Canada,"2,488.8800","109,696.8300",44.0748
1732,2024-12-18,TikTok Ads,Shopping,Fintech,USA,"3,488.9900","146,453.9200",41.9760
1631,2024-11-29,TikTok Ads,Search,E-commerce,Germany,"2,546.3400","106,309.3100",41.7498


Revenue share of these 5 rows: 0.0091


In [7]:
clean.to_csv(OUT, index=False)

check = pd.read_csv(OUT, parse_dates=["date", "month"])
print("Saved:", check.shape)
print(list(check.columns))
check.dtypes

Saved: (1800, 12)
['date', 'platform', 'campaign_type', 'industry', 'country', 'impressions', 'clicks', 'ad_spend', 'conversions', 'revenue', 'month', 'quarter']


date             datetime64[ns]
platform                 object
campaign_type            object
industry                 object
country                  object
impressions               int64
clicks                    int64
ad_spend                float64
conversions               int64
revenue                 float64
month            datetime64[ns]
quarter                  object
dtype: object

**Findings:**
- Rows in / out: 1800 / 1800 (no rows removed)
- Columns dropped: CTR, CPC, CPA, ROAS (pre-computed, row-level)
- Columns added: month, quarter
- Duplicate key combinations: 4 (reported, not removed)
- Outliers: kept; top 5 ROAS rows (41.7-49, all TikTok) = 0.91% of total revenue
- Aggregate ROAS 4.8775 vs mean of row ROAS 6.4504 (median 4.295)